# Module 2: The Engineer's Toolkit (T1–T7)
## Lab: Pipeline Construction — RFDiffusion → ProteinMPNN → AlphaFold

**Lagos Bio-Design Bootcamp** | [JobAiReady](https://github.com/JobAiReady/lagos-bio-design)

---

### Objective
Create a de novo protein backbone using RFDiffusion and generate a sequence for it using ProteinMPNN.

### Prerequisites
- Completed Module 1
- **GPU Runtime enabled** (Runtime → Change runtime type → T4 GPU)

### Deliverable
The SC (Self-Consistency) score plot showing < 2.0 Å RMSD.

> ⚠️ **Important:** This notebook requires a GPU runtime. Go to **Runtime → Change runtime type → T4 GPU** before running.

---
## Background & Key Concepts

Modern protein engineering follows a 3-tool pipeline that maps onto the 2025 Nature Reviews Bioengineering roadmap:

```
RFDiffusion  →  ProteinMPNN  →  AlphaFold/ESMFold
 (Generate)     (Sequence)       (Validate)
  backbone       design         self-consistency
  from noise     for fold        check (RMSD)
```

**RFDiffusion** is a diffusion model for protein structures. Like image generators (Stable Diffusion, DALL·E) that start from random noise and refine it into a picture, RFDiffusion starts from random 3D coordinates and denoises them into a valid protein backbone.

**ProteinMPNN** solves the *inverse folding* problem: given a 3D backbone, what amino acid sequence will fold into that exact shape? It bridges structure (what you designed) and sequence (what you can synthesize as DNA).

**Self-consistency check**: fold the designed sequence with AlphaFold/ESMFold and compare to the original backbone. **RMSD < 2.0 Å** = the design is likely to fold correctly in reality.

### Key Terms

| Term | Definition |
|------|-----------|
| **RFDiffusion** | Generative AI model that creates novel protein backbones by reversing a diffusion (noising) process |
| **ProteinMPNN** | Neural network that solves inverse folding — designing sequences that fold into a given 3D backbone |
| **Inverse Folding** | Given a target 3D structure, find a sequence that folds into it. The bridge between design and DNA synthesis |
| **RMSD** | Root Mean Square Deviation — measures structural similarity in Ångströms. <2.0 Å = successful design |
| **Self-Consistency** | Design a sequence → fold it → compare to intended backbone. Low RMSD = self-consistent |
| **Backbone** | The repeating N-Cα-C chain forming the structural skeleton of a protein |

---
## Setup: Install RFDiffusion & ProteinMPNN
This cell installs the required tools. It takes ~5 minutes on a fresh runtime.

In [ ]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), 'GPU not found! Go to Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# Install RFDiffusion
import os

if not os.path.exists('RFdiffusion'):
    !git clone https://github.com/RosettaCommons/RFdiffusion.git
    %cd RFdiffusion
    !pip install -q -e .
    
    # Download model weights
    os.makedirs('models', exist_ok=True)
    !wget -q -P models/ http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt
    print('RFDiffusion installed and weights downloaded.')
else:
    %cd RFdiffusion
    print('RFDiffusion already installed.')

In [ ]:
# Install ProteinMPNN
os.chdir('/content')

if not os.path.exists('ProteinMPNN'):
    !git clone https://github.com/dauparas/ProteinMPNN.git
    print('ProteinMPNN installed.')
else:
    print('ProteinMPNN already installed.')

# Install analysis dependencies
!pip install -q biopython matplotlib numpy

---
## Step 1: Backbone Generation with RFDiffusion
Generate a 100-residue de novo protein backbone with a specific fold topology.

**What's happening:** RFDiffusion starts from random noise (3D coordinates) and iteratively denoises them into a valid protein backbone — like sculpting a protein from static.

In [ ]:
os.chdir('/content/RFdiffusion')

# Generate a 100-residue backbone
!python scripts/run_inference.py \
    inference.output_prefix=/content/outputs/test_backbone \
    inference.model_directory_path=models/ \
    'contigmap.contigs=[100-100]' \
    inference.num_designs=3 \
    denoiser.noise_scale_ca=0 \
    denoiser.noise_scale_frame=0

In [ ]:
# Inspect the generated backbones
import glob
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
pdbs = sorted(glob.glob('/content/outputs/test_backbone_*.pdb'))

for pdb_path in pdbs:
    structure = parser.get_structure('bb', pdb_path)
    residue_count = sum(1 for _ in structure.get_residues())
    print(f'{os.path.basename(pdb_path)}: {residue_count} residues')

# Use the first design for the pipeline
backbone_pdb = pdbs[0]
print(f'\nUsing: {backbone_pdb}')

---
## Step 2: Sequence Design with ProteinMPNN
Feed the generated backbone into ProteinMPNN to find amino acid sequences that fold into this shape.

**What's happening:** ProteinMPNN solves the 'inverse folding' problem — given a 3D structure, what sequence of amino acids will fold into it?

In [ ]:
os.chdir('/content/ProteinMPNN')

# Run ProteinMPNN to design 5 sequences for the backbone
!python protein_mpnn_run.py \
    --pdb_path {backbone_pdb} \
    --out_folder /content/mpnn_results \
    --num_seq_per_target 5 \
    --sampling_temp 0.1 \
    --batch_size 1

In [ ]:
# Parse and display designed sequences
import glob

fasta_files = glob.glob('/content/mpnn_results/seqs/*.fa')
for fasta in fasta_files:
    with open(fasta) as f:
        lines = f.readlines()
    print(f'--- {os.path.basename(fasta)} ---')
    for i in range(0, len(lines), 2):
        header = lines[i].strip()
        seq = lines[i+1].strip() if i+1 < len(lines) else ''
        print(f'{header}')
        print(f'  {seq[:60]}...' if len(seq) > 60 else f'  {seq}')
    print()

---
## Step 3: Validation — Self-Consistency Check
Fold the designed sequence with AlphaFold/ESMFold and compare to the original backbone.

**Key metric:** RMSD (Root Mean Square Deviation)
- **< 1.0 Å**: Excellent — near-identical folds
- **1.0–2.0 Å**: Good — high structural similarity
- **> 2.0 Å**: Poor — design may not fold as intended

In [ ]:
# Use ESMFold for quick validation (faster than full AlphaFold)
!pip install -q fair-esm

import esm

# Load ESMFold model
esmfold = esm.pretrained.esmfold_v1()
esmfold = esmfold.eval().cuda()
print('ESMFold loaded on GPU.')

In [ ]:
# Fold the first designed sequence
with open(glob.glob('/content/mpnn_results/seqs/*.fa')[0]) as f:
    lines = f.readlines()

# Get the first designed sequence (skip the native sequence at index 0)
designed_seq = lines[3].strip() if len(lines) > 3 else lines[1].strip()
print(f'Folding sequence ({len(designed_seq)} residues)...')

with torch.no_grad():
    output = esmfold.infer_pdb(designed_seq)

# Save the predicted structure
with open('/content/folded_sequence.pdb', 'w') as f:
    f.write(output)

print('Folded structure saved to folded_sequence.pdb')

In [ ]:
# Calculate RMSD between original backbone and folded sequence
from Bio.PDB import PDBParser, Superimposer
import numpy as np

parser = PDBParser(QUIET=True)

# Load both structures
original = parser.get_structure('original', backbone_pdb)
predicted = parser.get_structure('predicted', '/content/folded_sequence.pdb')

# Extract CA atoms
def get_ca_atoms(structure):
    cas = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if 'CA' in residue:
                    cas.append(residue['CA'])
    return cas

ca_orig = get_ca_atoms(original)
ca_pred = get_ca_atoms(predicted)

# Align using the shorter length
n = min(len(ca_orig), len(ca_pred))
sup = Superimposer()
sup.set_atoms(ca_orig[:n], ca_pred[:n])

rmsd = sup.rms
print(f'\n=============================')
print(f'Self-Consistency RMSD: {rmsd:.2f} Å')
print(f'=============================')
print(f'Result: {"PASS ✓" if rmsd < 2.0 else "FAIL ✗"} (threshold: < 2.0 Å)')

In [ ]:
# Plot per-residue distance
import matplotlib.pyplot as plt

sup.apply(ca_pred[:n])
distances = [ca_orig[i] - ca_pred[i] for i in range(n)]

plt.figure(figsize=(12, 4))
plt.bar(range(n), distances, color=['#22c55e' if d < 2.0 else '#ef4444' for d in distances])
plt.axhline(y=2.0, color='red', linestyle='--', alpha=0.5, label='2.0 Å threshold')
plt.xlabel('Residue Index')
plt.ylabel('CA Distance (Å)')
plt.title(f'Self-Consistency Score — Overall RMSD: {rmsd:.2f} Å')
plt.legend()
plt.tight_layout()
plt.show()

---
## Summary

In this lab you completed the full de novo protein design pipeline:

1. **RFDiffusion** — Generated a novel 100-residue backbone from noise
2. **ProteinMPNN** — Designed amino acid sequences for the backbone (inverse folding)
3. **ESMFold/AlphaFold** — Validated the design via self-consistency check

This is the same pipeline used in state-of-the-art protein engineering labs worldwide.

**Next:** Module 3 — Binder Design & Controllable Hallucination